# Chapter 3: Data cleaning in health contexts

**Data Analysis with Python for Health Specialists** | 3 hours

---

## 1. Why data cleaning matters in health

Every health dataset has problems. Lab values get entered with wrong units. Patients appear twice under different IDs. Dates are recorded as `03/04/2023` --- is that March 4 or April 3?

Data cleaning is a **clinical reasoning task**. You must decide:
- Is this value *wrong* or just *unusual*?
- Should this record be *corrected*, *flagged*, or *removed*?
- Will my cleaning decisions *bias* the analysis?

> In a 2019 study of electronic health records, 18% of lab values had at least one quality issue --- duplicate entries, physiologically impossible values, or inconsistent units.

## 2. Missing data: types and mechanisms

Not all missing data is the same. The mechanism determines what you can do about it:

1. **MCAR -- Missing Completely At Random.** The missingness has nothing to do with the data. Example: a lab tube is dropped.
2. **MAR -- Missing At Random.** Missingness depends on *observed* variables. Example: younger patients are less likely to have cholesterol measured.
3. **MNAR -- Missing Not At Random.** Missingness depends on the *unobserved* value itself. Example: very sick patients miss follow-up appointments.

> **Warning:** MNAR is common in health data. If you drop patients with missing follow-up, you systematically exclude the sickest patients.

## 3. Detecting missing values with pandas

In [ ]:
import pandas as pd
import numpy as np

# Create a clinical dataset with missing values
data = {
    "patient_id": ["P001", "P002", "P003", "P004", "P005", "P006"],
    "age": [45, 67, 34, np.nan, 71, 52],
    "sex": ["M", "F", "F", "M", np.nan, "F"],
    "systolic_bp": [128, 158, np.nan, 145, 162, np.nan],
    "hba1c": [5.4, 7.8, 5.1, np.nan, 8.2, 6.3],
    "smoking": ["Never", np.nan, "Current", "Former", np.nan, "Never"]
}
df = pd.DataFrame(data)

# Count missing values per column
print(df.isnull().sum())

# Percentage missing per column
print((df.isnull().sum() / len(df) * 100).round(1))

# Rows with any missing value
print(f"Rows with missing data: {df.isnull().any(axis=1).sum()} / {len(df)}")

### Visualizing missingness with missingno

In [ ]:
# !pip install missingno
import missingno as msno
import matplotlib.pyplot as plt

# Matrix plot: white bars = missing
msno.matrix(df)
plt.title("Missing data pattern")
plt.tight_layout()
plt.show()

# Bar chart: count of non-null values per column
msno.bar(df)
plt.tight_layout()
plt.show()

# Heatmap: correlations between missingness of different columns
msno.heatmap(df)
plt.tight_layout()
plt.show()

In [ ]:
# Check which columns are missing together
missing_pairs = df.isnull().corr()
print(missing_pairs)

## 4. Imputation strategies

- **Drop rows** if missingness is MCAR and the missing fraction is small (<5%).
- **Drop columns** if >50% of values are missing and the variable is not critical.
- **Impute** when you need to retain sample size.
- **Flag + sensitivity analysis** for MNAR data.

In [ ]:
# Drop rows with any missing value
df_complete = df.dropna()
print(f"Rows remaining: {len(df_complete)} / {len(df)}")

# Drop rows only if specific columns are missing
df_partial = df.dropna(subset=["age", "systolic_bp"])

### Mean/median imputation for continuous variables

Use **median** rather than mean for lab values (robust to outliers).

In [ ]:
# Median imputation for lab values (robust to outliers)
df["systolic_bp"] = df["systolic_bp"].fillna(df["systolic_bp"].median())
df["hba1c"] = df["hba1c"].fillna(df["hba1c"].median())

# Mean imputation for age (approximately symmetric distribution)
df["age"] = df["age"].fillna(df["age"].mean())

In [ ]:
# Mode imputation for sex and smoking status
df["sex"] = df["sex"].fillna(df["sex"].mode()[0])
df["smoking"] = df["smoking"].fillna(df["smoking"].mode()[0])

### Group-specific imputation

Sometimes the right imputation value depends on a group. Imputing systolic BP with the overall median ignores that older patients have higher BP.

In [ ]:
# Impute systolic_bp with the median for the patient's age group
df["age_group"] = pd.cut(df["age"], bins=[0, 40, 60, 100],
                         labels=["<40", "40-59", "60+"])
df["systolic_bp"] = df.groupby("age_group")["systolic_bp"].transform(
    lambda x: x.fillna(x.median())
)

### KNN imputation

K-nearest neighbors finds the k most similar patients and uses their values to fill in missing ones.

In [ ]:
from sklearn.impute import KNNImputer

# Select numeric columns for KNN imputation
numeric_cols = ["age", "systolic_bp", "hba1c"]
imputer = KNNImputer(n_neighbors=5)
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

### Creating a missingness indicator

For important variables, create a flag so you can later test whether missingness is associated with outcomes.

In [ ]:
# Before imputing, create a flag
df["bp_was_missing"] = df["systolic_bp"].isnull().astype(int)

# Then impute
df["systolic_bp"] = df["systolic_bp"].fillna(df["systolic_bp"].median())

## 5. Handling duplicates

In [ ]:
# Load a dataset with deliberate duplicates
patients = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003", "P002", "P004", "P003"],
    "visit_date": ["2023-01-15", "2023-01-16", "2023-01-17",
                   "2023-01-16", "2023-01-18", "2023-02-20"],
    "systolic_bp": [128, 158, 112, 158, 145, 118],
    "diagnosis": ["HTN", "T2DM", "None", "T2DM", "HTN", "None"]
})

# Exact duplicates (all columns identical)
print(f"Exact duplicates: {patients.duplicated().sum()}")
print(patients[patients.duplicated(keep=False)])  # show all copies

# Duplicates based on patient_id only
print(f"\nDuplicate patient IDs: {patients.duplicated(subset='patient_id').sum()}")

In [ ]:
# Remove exact duplicates
patients_clean = patients.drop_duplicates()
print(f"After removing exact duplicates: {len(patients_clean)} rows")

# Keep only the first visit per patient
first_visits = patients.drop_duplicates(subset="patient_id", keep="first")
print(f"First visits only: {len(first_visits)} rows")

# Keep only the last visit per patient
last_visits = patients.drop_duplicates(subset="patient_id", keep="last")

## 6. Inconsistent data

### ICD code formats

In [ ]:
diagnoses = pd.Series(["E11.9", "e11.9", "E119", "E11.9 ", " e11.9"])

# Standardize: uppercase, strip whitespace, ensure dot format
cleaned = (diagnoses
           .str.strip()
           .str.upper()
           .str.replace(r"^([A-Z]\d{2})(\d+)$", r"\1.\2", regex=True))
print(cleaned)

### Date formats

In [ ]:
dates = pd.Series(["2023-01-15", "01/15/2023", "15-Jan-2023",
                   "Jan 15, 2023", "20230115"])

# pd.to_datetime handles multiple formats automatically
parsed = pd.to_datetime(dates, format="mixed", dayfirst=False)
print(parsed)

# Standardize to ISO format
print(parsed.dt.strftime("%Y-%m-%d"))

### Unit conversions and standardizing text fields

In [ ]:
# Standardize glucose to mmol/L
# Assume values > 30 are in mg/dL (glucose in mmol/L is rarely > 30)
def standardize_glucose(value, unit=None):
    """Convert glucose to mmol/L. Infer unit if not provided."""
    if unit == "mg/dL" or (unit is None and value > 30):
        return value / 18.018
    return value  # already in mmol/L

# Test it
print(f"126 mg/dL -> {standardize_glucose(126):.1f} mmol/L")
print(f"7.0 mmol/L -> {standardize_glucose(7.0):.1f} mmol/L")

In [ ]:
# Sex/gender field with inconsistent entries
sex_col = pd.Series(["Male", "male", "M", "m", "MALE", "Female",
                     "female", "F", "f", "FEMALE"])

# Map to standard values
sex_mapping = {"male": "M", "m": "M", "female": "F", "f": "F"}
sex_clean = sex_col.str.strip().str.lower().map(sex_mapping)
print(sex_clean)

## 7. Data validation

After cleaning, validate that your data makes clinical sense.

In [ ]:
# Define clinically plausible ranges
ranges = {
    "age": (0, 120),
    "systolic_bp": (60, 300),
    "diastolic_bp": (30, 200),
    "heart_rate": (20, 300),
    "hba1c": (2.0, 20.0),
    "glucose_mgdl": (10, 1500),
    "bmi": (8, 80),
    "temperature_c": (25, 45),
}

def validate_range(df, column, low, high):
    """Flag values outside the plausible range."""
    out_of_range = df[(df[column] < low) | (df[column] > high)]
    if len(out_of_range) > 0:
        print(f"WARNING: {len(out_of_range)} values in '{column}' "
              f"outside [{low}, {high}]")
        print(out_of_range[["patient_id", column]])
    return out_of_range

# Apply range checks
for col, (low, high) in ranges.items():
    if col in df.columns:
        validate_range(df, col, low, high)

### Cross-field validation and automated quality report

In [ ]:
def data_quality_report(df):
    """Generate a summary of data quality issues."""
    report = []
    for col in df.columns:
        n_missing = df[col].isnull().sum()
        pct_missing = n_missing / len(df) * 100
        n_unique = df[col].nunique()

        row = {
            "column": col,
            "dtype": str(df[col].dtype),
            "n_missing": n_missing,
            "pct_missing": round(pct_missing, 1),
            "n_unique": n_unique,
        }

        if df[col].dtype in ["float64", "int64"]:
            row["min"] = df[col].min()
            row["max"] = df[col].max()
            row["mean"] = round(df[col].mean(), 2)

        report.append(row)

    return pd.DataFrame(report)

quality = data_quality_report(df)
print(quality.to_string(index=False))

## 8. A complete cleaning pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# -----------------------------------------------------------
# Step 1: Load data
# -----------------------------------------------------------
url = ("https://raw.githubusercontent.com/datasets/"
       "gapminder/main/data/gapminder.csv")
df = pd.read_csv(url)
print(f"Raw data: {df.shape}")

# -----------------------------------------------------------
# Step 2: Standardize column names
# -----------------------------------------------------------
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# -----------------------------------------------------------
# Step 3: Remove exact duplicates
# -----------------------------------------------------------
n_before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {n_before - len(df)}")

# -----------------------------------------------------------
# Step 4: Assess missing data
# -----------------------------------------------------------
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print("Missing percentages:")
print(missing_pct[missing_pct > 0])

# -----------------------------------------------------------
# Step 5: Validate ranges
# -----------------------------------------------------------
# Life expectancy should be between 20 and 90
outliers = df[(df["lifeexp"] < 20) | (df["lifeexp"] > 90)]
print(f"Life expectancy outliers: {len(outliers)}")

# -----------------------------------------------------------
# Step 6: Save cleaned data
# -----------------------------------------------------------
df.to_csv("gapminder_cleaned.csv", index=False)
print(f"Cleaned data saved: {df.shape}")

## Exercises

1. Generate a data quality report for the messy dataset below: count missing values, check data types, compute summary statistics.
2. Identify and remove exact duplicate rows.
3. Standardize the `sex` column to "M" and "F". Handle "X" and NaN.
4. Validate `age` (flag impossible values like -3 and 200), `systolic_bp` (450 is impossible), and check diastolic < systolic.
5. Fix `glucose_mgdl`: values 7.2, 6.8, 5.5 are in mmol/L --- convert to mg/dL (multiply by 18.018).
6. Fix `hba1c`: value 55 is in mmol/mol (IFCC) --- convert using NGSP = 0.0915 * IFCC + 2.15.
7. Standardize `visit_date` to ISO format and `icd_code` to uppercase with dot separator.
8. Impute remaining missing values and save the cleaned dataset.

In [ ]:
# Messy clinical data with deliberate errors
messy = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003", "P004", "P002",
                   "P005", "P006", "P007", "P008", "P009"],
    "age": [45, 67, -3, 52, 67, 200, 38, np.nan, 71, 44],
    "sex": ["M", "F", "Female", "m", "F",
            "Male", np.nan, "F", "X", "M"],
    "systolic_bp": [128, 158, 112, 450, 158,
                    135, 122, np.nan, 162, 118],
    "diastolic_bp": [82, 160, 74, 92, 160,
                     88, 78, np.nan, 98, 76],
    "glucose_mgdl": [95, 7.2, 110, 180, 95,
                     88, 102, 6.8, 220, 5.5],
    "hba1c": [5.4, 7.8, 5.1, 6.3, 7.8,
              np.nan, 5.8, 6.1, 55, 5.2],
    "visit_date": ["2023-01-15", "01/16/2023", "2023-01-17",
                   "2023/01/18", "2023-01-16",
                   "15-Jan-2023", "2023-01-20", "2023-01-21",
                   "2023-01-22", "Jan 23, 2023"],
    "icd_code": ["I10", "E119", "e11.9", "I10 ", " i10",
                 "E11.9", "I10", "e119", "I10", "E11.9"]
})
messy

In [ ]:
# Exercise 1: your code here


In [ ]:
# Exercise 2: your code here


In [ ]:
# Exercise 3: your code here


In [ ]:
# Exercise 4: your code here


In [ ]:
# Exercise 5: your code here


In [ ]:
# Exercise 6: your code here


In [ ]:
# Exercise 7: your code here


In [ ]:
# Exercise 8: your code here
